# Passes

Detects pass events from `ball_frame_table`'s carrier transitions and
reports **all passes** and **completed passes** for both teams. Reads only
`player_frame_table` / `ball_frame_table` (built by `match_frame_table.ipynb`)
-- never the raw per-stage caches.

**Definition used here** (this is a proxy, not ground truth -- worth being
explicit about since there's no whistle/event feed to check against):
a *pass* is a transition of ball possession from one player's continuous
carrier segment to a different player's, close enough in time to be one
continuous phase of play. A pass is **completed** if the receiver is on the
same team as the passer, and a **turnover** if not. Consecutive frames where
the *same* player regains the carrier tag after a brief tracking blip are
merged into one segment, not counted as a pass.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import paths
from ball_tracker import BallTrackerConfig, CarrierConfig  # FIX: source shared thresholds from here, not separate hardcoded literals below

carrier_cfg = CarrierConfig()
ball_cfg = BallTrackerConfig()

for p in (paths.PLAYER_FRAME_TABLE_CACHE_PATH, paths.BALL_FRAME_TABLE_CACHE_PATH):
    if not Path(p).exists():
        raise FileNotFoundError(
            f"{p} not found -- run match_frame_table.ipynb first, it builds both frame tables."
        )

player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)

print("player_frame_table:", player_frame_table.shape)
print("ball_frame_table:  ", ball_frame_table.shape)

player_frame_table: (70396, 10)
ball_frame_table:   (3001, 10)


## Team lookup

A track's team is effectively constant across the match, so this collapses
`player_frame_table` down to one `track_id -> team` mapping (majority vote,
in case of a rare mis-assigned frame). Referees have no team and are
excluded -- they never appear as a carrier anyway, since `ball_tracker`'s
carrier assigner only considers player/goalkeeper tracks.

In [10]:
team_by_id = (
    player_frame_table.dropna(subset=["team"])
    .groupby("track_id")["team"]
    .agg(lambda s: s.mode().iat[0])
    .astype(int)
    .to_dict()
)
print(f"{len(team_by_id)} tracks with a team assigned")

22 tracks with a team assigned


## Possession segments

Collapse `ball_frame_table.carrier_track_id` into runs: `(track_id,
start_frame, end_frame, n_frames)`. A new segment starts whenever the
carrier changes **or** whenever there's a gap in frame coverage (carrier
was unassigned in between) -- so two same-player runs separated by a gap
stay two segments unless later merged by the pass-linking step below, and
a same-player run interrupted by a single missing frame (no gap) is *not*
artificially split.

In [11]:
def build_possession_segments(ball_frame_table):
    df = ball_frame_table[["frame_idx", "carrier_track_id"]].dropna(subset=["carrier_track_id"]).copy()
    df["carrier_track_id"] = df["carrier_track_id"].astype(int)
    if df.empty:
        return pd.DataFrame(columns=["track_id", "start_frame", "end_frame", "n_frames"])

    changed = df["carrier_track_id"] != df["carrier_track_id"].shift()
    gapped = df["frame_idx"] != (df["frame_idx"].shift() + 1)
    segment_id = (changed | gapped).cumsum()

    segments = (
        df.groupby(segment_id)
        .agg(track_id=("carrier_track_id", "first"),
             start_frame=("frame_idx", "min"),
             end_frame=("frame_idx", "max"),
             n_frames=("frame_idx", "size"))
        .reset_index(drop=True)
    )
    return segments


# FIX: this used to be hardcoded to 3 with a comment claiming it matched
# CarrierConfig.min_frames_to_switch -- the actual config value is 7, so the
# literal and the comment had silently drifted apart. Sourced directly from
# the config now so the two can't disagree again.
MIN_SEGMENT_FRAMES = carrier_cfg.min_frames_to_switch  # anything shorter than the carrier
                                                         # assigner's own switch hysteresis is noise

segments = build_possession_segments(ball_frame_table)
n_before = len(segments)
segments = segments[segments["n_frames"] >= MIN_SEGMENT_FRAMES].reset_index(drop=True)
print(f"Possession segments: {n_before} -> {len(segments)} after dropping runs shorter than {MIN_SEGMENT_FRAMES} frames")
segments.head()

Possession segments: 41 -> 41 after dropping runs shorter than 3 frames


,track_id,start_frame,end_frame,n_frames
0,13,9,64,56
1,18,90,109,20
2,9,132,167,36
3,21,211,251,41
4,23,252,286,35


## Pass events

Link adjacent segments into pass events. Two segments are linked as a pass
when: the carrier actually changed (not the same player reappearing after
a blip), and the gap between the passer's last frame and the receiver's
first frame is short enough to plausibly be one continuous pass rather than
a stoppage or restart.

`MAX_GAP_FRAMES` is deliberately generous (3s @ 25fps) -- long balls and
crosses routinely lose ball-tracking mid-flight (small, fast, sometimes
airborne), and a long pass shouldn't be thrown out just because the
Kalman/RTS tracker had a gap during it. A multi-second gap, on the other
hand, is far more likely a throw-in, goal kick, or other restart than a
single pass.

In [12]:
# FIX: this was hardcoded to 75 with a comment claiming "~3s @ ball_tracker.FPS"
# rather than actually being derived from it -- correct today (25fps x 3s = 75)
# but would silently go stale if fps ever changed. Computed from the real fps
# constant now.
MAX_GAP_SECONDS = 3.0  # long balls/crosses routinely lose ball-tracking mid-flight; a multi-second
                        # gap is far more likely a restart (throw-in, goal kick, etc.) than one pass
MAX_GAP_FRAMES = round(MAX_GAP_SECONDS * ball_cfg.fps)


def build_pass_events(segments, team_by_id, max_gap_frames=MAX_GAP_FRAMES):
    events = []
    for i in range(len(segments) - 1):
        passer = segments.iloc[i]
        receiver = segments.iloc[i + 1]

        if passer["track_id"] == receiver["track_id"]:
            continue  # same player -- a tracking blip mid-possession, not a pass

        gap = receiver["start_frame"] - passer["end_frame"]
        if gap > max_gap_frames:
            continue  # too long to attribute to one continuous pass

        passer_team = team_by_id.get(passer["track_id"])
        receiver_team = team_by_id.get(receiver["track_id"])
        if passer_team is None or receiver_team is None:
            continue  # can't classify outcome without both teams known

        events.append({
            "pass_frame": int(passer["end_frame"]),
            "receive_frame": int(receiver["start_frame"]),
            "gap_frames": int(gap),
            "passer_id": int(passer["track_id"]),
            "receiver_id": int(receiver["track_id"]),
            "passer_team": passer_team,
            "receiver_team": receiver_team,
            "completed": passer_team == receiver_team,
        })

    return pd.DataFrame(events, columns=[
        "pass_frame", "receive_frame", "gap_frames",
        "passer_id", "receiver_id", "passer_team", "receiver_team", "completed",
    ])


pass_events = build_pass_events(segments, team_by_id)
print(f"{len(pass_events)} pass events detected")
pass_events.head(10)

38 pass events detected


,pass_frame,receive_frame,gap_frames,passer_id,receiver_id,passer_team,receiver_team,completed
0,64,90,26,13,18,1,1,True
1,109,132,23,18,9,1,1,True
2,167,211,44,9,21,1,1,True
3,251,252,1,21,23,1,1,True
4,286,287,1,23,15,1,0,False
5,324,325,1,15,12,0,0,True
6,497,525,28,12,10,0,0,True
7,601,619,18,10,19,0,0,True
8,646,663,17,19,16,0,0,True
9,881,901,20,12,10,0,0,True


## Sanity checks

In [13]:
print(f"Total pass events      -> {len(pass_events)}")
print(f"Completed               -> {pass_events['completed'].sum()} ({pass_events['completed'].mean()*100:.1f}%)")
print(f"Turnovers                -> {(~pass_events['completed']).sum()}")
print()
print("gap_frames distribution (time between passer losing it and receiver gaining it):")
print(pass_events["gap_frames"].describe())
print()
print("Passes per passer team:")
print(pass_events["passer_team"].value_counts().sort_index())

Total pass events      -> 38
Completed               -> 33 (86.8%)
Turnovers                -> 5

gap_frames distribution (time between passer losing it and receiver gaining it):
count    38.000000
mean     18.526316
std      13.904572
min       1.000000
25%       1.000000
50%      18.500000
75%      29.500000
max      45.000000
Name: gap_frames, dtype: float64

Passes per passer team:
passer_team
0    23
1    15
Name: count, dtype: int64


## All passes and completed passes by team

`attempted` counts every pass event where that team had the ball at the
start of the transition, regardless of outcome. `completed` is the
same-team-receiver subset.

In [14]:
team_summary = (
    pass_events.groupby("passer_team")
    .agg(attempted=("completed", "size"), completed=("completed", "sum"))
    .reset_index()
    .rename(columns={"passer_team": "team"})
)
team_summary["completion_pct"] = (team_summary["completed"] / team_summary["attempted"] * 100).round(1)

print("Per team:")
print(team_summary.to_string(index=False))
print()
print(f"All passes (both teams)       -> {team_summary['attempted'].sum()}")
print(f"Completed passes (both teams) -> {team_summary['completed'].sum()}")
team_summary

Per team:
 team  attempted  completed  completion_pct
    0         23         20            87.0
    1         15         13            86.7

All passes (both teams)       -> 38
Completed passes (both teams) -> 33


,team,attempted,completed,completion_pct
0,0,23,20,87.0
1,1,15,13,86.7


## Top passers by team

Same `pass_events` table, grouped one level deeper: `(passer_team,
passer_id)` instead of just `passer_team`. `TOP_N_PASSERS` caps how many
players are shown per team, ranked by attempted passes.

In [15]:
TOP_N_PASSERS = 5

passer_stats = (
    pass_events.groupby(["passer_team", "passer_id"])
    .agg(attempted=("completed", "size"), completed=("completed", "sum"))
    .reset_index()
)
passer_stats["completion_pct"] = (passer_stats["completed"] / passer_stats["attempted"] * 100).round(1)

top_passers = (
    passer_stats.sort_values(["passer_team", "attempted"], ascending=[True, False])
    .groupby("passer_team")
    .head(TOP_N_PASSERS)
    .reset_index(drop=True)
)

for team, group in top_passers.groupby("passer_team"):
    print(f"\nTop passers -- team {team}:")
    print(group[["passer_id", "attempted", "completed", "completion_pct"]].to_string(index=False))

top_passers


Top passers -- team 0:
 passer_id  attempted  completed  completion_pct
        10          7          7           100.0
        12          3          3           100.0
        22          3          1            33.3
         4          2          2           100.0
         8          2          2           100.0

Top passers -- team 1:
 passer_id  attempted  completed  completion_pct
        18          3          3           100.0
        21          3          3           100.0
        23          3          2            66.7
         9          2          2           100.0
        13          2          1            50.0


,passer_team,passer_id,attempted,completed,completion_pct
0,0,10,7,7,100.0
1,0,12,3,3,100.0
2,0,22,3,1,33.3
3,0,4,2,2,100.0
4,0,8,2,2,100.0
5,1,18,3,3,100.0
6,1,21,3,3,100.0
7,1,23,3,2,66.7
8,1,9,2,2,100.0
9,1,13,2,1,50.0


## Event classification: duel, shot, or pass

Replaces the earlier zone-based shot guess. A carrier transition to a
different track isn't just "pass" vs "turnover" -- it can be a **duel**
(tackled/dispossessed -- the new carrier was standing right next to the old
one, no real delivery happened), a **shot** (ball velocity pointed at the
goal mouth just before the transition, whether or not a receiver ever picks
it up), or a genuine pass/turnover.

Two signals, both derived from data already in `player_frame_table` /
`ball_frame_table` -- no new upstream columns needed:

- **Contest distance**: distance between the passer's and receiver's own
  pitch positions (not the ball) at the transition. A duel means these two
  players were adjacent when possession flipped; a real pass means the
  receiver was somewhere else on the pitch, not marking the passer.
- **Shot angle**: direction of the ball's velocity (trailing-window diff on
  `ball_pitch_x/y`) in the frames right before the transition, compared to
  the vector from the ball to the opponent's goal (using `attack_direction`,
  already computed). A shot points at goal; a pass to a teammate's feet
  generally doesn't.

This also picks up **shots that go wide/out**, which today are invisible --
the segment just ends and the gap-length filter silently drops the event.
Those are handled separately below by scanning segments that never became a
"passer" in `pass_events` at all, not by inferring them from a receiver.

Still open: separating a genuine interception from a misplaced pass once
both the duel and shot cases are ruled out -- that still needs trajectory
curvature (did the ball change direction abruptly, meaning someone read and
cut it out, vs. arrived where a teammate was already moving to meet it),
which isn't computed yet. Distances/angles below are tunable constants --
worth checking against `ball_speed`/`contest_distance`/`shot_angle_deg`
distributions on real data (`pass_events[...].describe()`) before trusting
the thresholds, rather than assuming these numbers are right out of the
gate.

In [ ]:
DUEL_DISTANCE_M = 2.5       # passer/receiver this close at the transition -> a duel, not a delivery
SHOT_ANGLE_DEG = 30.0       # ball velocity within this many degrees of the goal-ward vector -> shot
MIN_BALL_SPEED_MPS = 8.0    # below this, a "shot angle" match is more likely a slow rolling ball than a strike

pitch_length, pitch_width = hom_cfg.pitch_length, hom_cfg.pitch_width if False else (None, None)
from homography import HomographyConfig
hom_cfg = HomographyConfig()
pitch_length, pitch_width = hom_cfg.pitch_length, hom_cfg.pitch_width

attack_direction_by_team = (
    player_frame_table.dropna(subset=["team", "attack_direction"])
    .groupby("team")["attack_direction"]
    .agg(lambda s: s.mode().iat[0])
    .astype(int)
    .to_dict()
)

ball_pos = ball_frame_table.set_index("frame_idx")[["ball_pitch_x", "ball_pitch_y"]]
player_pos = player_frame_table.set_index(["track_id", "frame_idx"])[["pitch_x", "pitch_y"]]


def ball_velocity(frame_idx, window=5, fps=ball_cfg.fps):
    """Trailing-window velocity ending at frame_idx: (vx, vy, speed) in m/s, pitch coords.
    NaN if fewer than 2 valid ball positions in the window."""
    sub = ball_pos.loc[max(frame_idx - window, 0):frame_idx].dropna()
    if len(sub) < 2:
        return np.nan, np.nan, np.nan
    (f0, x0, y0), (f1, x1, y1) = (sub.index[0], *sub.iloc[0]), (sub.index[-1], *sub.iloc[-1])
    dt = (f1 - f0) / fps
    if dt <= 0:
        return np.nan, np.nan, np.nan
    return (x1 - x0) / dt, (y1 - y0) / dt, np.hypot(x1 - x0, y1 - y0) / dt


def player_xy(track_id, frame_idx, tolerance=5):
    """Nearest available (pitch_x, pitch_y) for track_id within `tolerance` frames of frame_idx."""
    for df in (frame_idx + off for off in range(0, tolerance + 1) for frame_idx in [frame_idx]):
        pass
    for off in range(0, tolerance + 1):
        for f in ({frame_idx - off, frame_idx + off} if off else {frame_idx}):
            if (track_id, f) in player_pos.index:
                row = player_pos.loc[(track_id, f)]
                if pd.notna(row["pitch_x"]):
                    return row["pitch_x"], row["pitch_y"]
    return np.nan, np.nan


def shot_angle_deg(vx, vy, ball_x, ball_y, passer_team):
    """Angle between ball velocity and the vector to the opponent's goal. NaN if inputs missing."""
    direction = attack_direction_by_team.get(passer_team)
    if direction is None or np.isnan(vx) or np.isnan(ball_x):
        return np.nan
    goal_x = pitch_length if direction == 1 else 0.0
    goal_y = pitch_width / 2
    to_goal = np.array([goal_x - ball_x, goal_y - ball_y])
    vel = np.array([vx, vy])
    denom = np.linalg.norm(to_goal) * np.linalg.norm(vel)
    if denom == 0:
        return np.nan
    cos_angle = np.clip(np.dot(to_goal, vel) / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))


def classify_event(row):
    if row["passer_team"] == row["receiver_team"]:
        return "pass"

    px, py = player_xy(row["passer_id"], row["pass_frame"])
    rx, ry = player_xy(row["receiver_id"], row["receive_frame"])
    contest_distance = np.hypot(rx - px, ry - py) if not (np.isnan(px) or np.isnan(rx)) else np.nan
    if not np.isnan(contest_distance) and contest_distance <= DUEL_DISTANCE_M:
        return "dispossessed"

    vx, vy, speed = ball_velocity(row["pass_frame"])
    bx, by = ball_pos.loc[row["pass_frame"]] if row["pass_frame"] in ball_pos.index else (np.nan, np.nan)
    angle = shot_angle_deg(vx, vy, bx, by, row["passer_team"])
    if not np.isnan(angle) and angle <= SHOT_ANGLE_DEG and speed >= MIN_BALL_SPEED_MPS:
        return "shot_on_target"  # reached an opposing player -- saved/blocked rather than scored, can't tell which without a goal-line/score signal

    return "turnover"  # different team, not a duel, not shot-angle -- best-effort label; could still be a misplaced pass or a clean interception


pass_events["event_type"] = pass_events.apply(classify_event, axis=1)
print(pass_events["event_type"].value_counts())

### Shots (and clearances) with no receiver

Segments that never appear as a `passer_id` in `pass_events` -- because
nothing picked the ball up within `MAX_GAP_FRAMES`, or it's the very last
segment in the clip -- are exactly where a shot that misses, or a clearance
that goes out of play, would show up. Today the gap filter drops them
silently; this recovers them and classifies by the same ball-velocity/angle
logic, plus a boundary check for "went out of play" vs. genuinely
unresolved (tracking loss with a non-committal ball direction).

In [ ]:
BOUNDARY_MARGIN_M = 2.0  # ball's last known position within this of a pitch edge -> likely out of play

linked_passer_ends = set(zip(pass_events["passer_id"], pass_events["pass_frame"]))

unresolved_rows = []
for _, seg in segments.iterrows():
    if (seg["track_id"], seg["end_frame"]) in linked_passer_ends:
        continue  # already accounted for as a pass/duel/shot event above

    team = team_by_id.get(seg["track_id"])
    if team is None:
        continue

    vx, vy, speed = ball_velocity(seg["end_frame"])
    bx, by = ball_pos.loc[seg["end_frame"]] if seg["end_frame"] in ball_pos.index else (np.nan, np.nan)
    angle = shot_angle_deg(vx, vy, bx, by, team)

    if not np.isnan(angle) and angle <= SHOT_ANGLE_DEG and speed >= MIN_BALL_SPEED_MPS:
        outcome = "shot_off_target"
    elif not np.isnan(bx) and (
        bx <= BOUNDARY_MARGIN_M or bx >= pitch_length - BOUNDARY_MARGIN_M
        or by <= BOUNDARY_MARGIN_M or by >= pitch_width - BOUNDARY_MARGIN_M
    ):
        outcome = "out_of_play"
    else:
        outcome = "unresolved"

    unresolved_rows.append({
        "end_frame": int(seg["end_frame"]),
        "track_id": int(seg["track_id"]),
        "team": int(team),
        "ball_speed": speed,
        "shot_angle_deg": angle,
        "outcome": outcome,
    })

no_receiver_events = pd.DataFrame(unresolved_rows)
print(no_receiver_events["outcome"].value_counts() if len(no_receiver_events) else "no unresolved segments")
no_receiver_events.head()

## Cache as parquet

Same load-or-build pattern as every other stage.

In [8]:
FORCE_REBUILD_PASSES = False


def get_or_build_pass_events(force_rebuild=FORCE_REBUILD_PASSES):
    cache_path = Path(paths.PASS_EVENTS_CACHE_PATH)
    if cache_path.exists() and not force_rebuild:
        print(f"\u2705 Loaded pass events from cache.")
        return pd.read_parquet(cache_path)

    segments = build_possession_segments(ball_frame_table)
    segments = segments[segments["n_frames"] >= MIN_SEGMENT_FRAMES].reset_index(drop=True)
    events = build_pass_events(segments, team_by_id)

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    events.to_parquet(cache_path, index=False)
    print(f"\U0001F4BE Saved pass events to cache.")
    return events


pass_events = get_or_build_pass_events()

💾 Saved pass events to cache.


## Next steps

`pressure.py` and `possession.py` can both read `pass_events` alongside
`player_frame_table` / `ball_frame_table` -- e.g. pressure that immediately
follows a turnover (a counter-press) is a distinct, interesting stat once
both tables exist.